# TravelTide – Data Exploration

## 1. Project Objective

This notebook is dedicated to the initial exploration and understanding of the TravelTide dataset.

### Objectives

- Load and inspect the source data.
- Understand the dataset structure and data types.
- Identify missing values and potential data quality issues.
- Explore relevant variables and their distributions.
- Prepare the data for subsequent cleaning and analysis.

### Data Management Principles

- Original source data must remain unchanged.
- All data cleaning and transformations must be performed on working copies.
- Intermediate and final datasets must be stored separately.

## 2. Environment Setup

This section prepares the Python environment for the TravelTide data exploration.

### 2.1 Import Libraries

Import the required Python libraries for data processing, numerical calculations, and data visualization.

In [10]:
# 2.1 Import libraries for data exploration

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configure pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

# Configure matplotlib
plt.style.use("default")

print("TravelTide data exploration environment is ready.")


TravelTide data exploration environment is ready.


## 3. Project Structure and Paths

This section defines the project directory structure and establishes reusable paths for data access.

### 3.1 Define Project Paths

Define the project root and all relevant data directories using pathlib.

All paths are derived from the project root to ensure consistent and portable file access.

In [11]:
# 3.1 Import Path for cross-platform file handling
from pathlib import Path

# Define project root directory
PROJECT_ROOT = Path.cwd().parent

# Define data directories
DATA_DIR = PROJECT_ROOT / "data"
INPUT_DIR = DATA_DIR / "input"
ORIGINAL_DIR = INPUT_DIR / "original"
VERSIONS_DIR = INPUT_DIR / "versions"

WORKING_DIR = DATA_DIR / "working"
CLEANING_DIR = WORKING_DIR / "cleaning"
TRANSFORMATIONS_DIR = WORKING_DIR / "transformations"
ANALYSIS_DIR = WORKING_DIR / "analysis"

OUTPUT_DIR = DATA_DIR / "output"
REPORTS_DIR = OUTPUT_DIR / "reports"
FINAL_DATASETS_DIR = OUTPUT_DIR / "final_datasets"

# Display project root
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

Project root: d:\MSIT_Schulungsunterlagen\Projekte\TravelTide
Data directory: d:\MSIT_Schulungsunterlagen\Projekte\TravelTide\data


### 3.2 Verify Project Directories

Verify that all required project data directories exist before starting the data exploration.

In [12]:
# 3.2 Verify project data directories

directories = {
    "Original data": ORIGINAL_DIR,
    "Data cleaning": CLEANING_DIR,
    "Data transformations": TRANSFORMATIONS_DIR,
    "Data analysis": ANALYSIS_DIR,
    "Reports": REPORTS_DIR,
    "Final datasets": FINAL_DATASETS_DIR,
}

for name, path in directories.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{name}: {status}")

Original data: OK
Data cleaning: OK
Data transformations: OK
Data analysis: OK
Reports: OK
Final datasets: OK


### 3.3 Display Environment Information

Display the Python version, installed library versions, active Python interpreter, and project root directory.

In [13]:
# 3.3 Display project environment information

import sys
import pandas as pd
import numpy as np
import matplotlib

print("TravelTide Environment Information")
print("-----------------------------------")
print(f"Python version: {sys.version.split()[0]}")
print(f"Python executable: {sys.executable}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Project root: {PROJECT_ROOT}")

TravelTide Environment Information
-----------------------------------
Python version: 3.14.6
Python executable: d:\MSIT_Schulungsunterlagen\Projekte\TravelTide\.venv\Scripts\python.exe
NumPy version: 2.5.4
Pandas version: 3.1.0
Matplotlib version: 3.11.2
Project root: d:\MSIT_Schulungsunterlagen\Projekte\TravelTide


## 4. Database Connection and Local Data Inventory

This section establishes the connection to the central Neon PostgreSQL database and manages local copies of the original source data.

The objective is to ensure that all required database tables are available locally while protecting existing original copies from accidental overwriting.

### Data Management Principles

- Neon is the central source of truth.
- Original database tables must never be modified by the analysis process.
- Existing local original copies must never be overwritten automatically.
- Only tables without an existing local copy are downloaded.
- Local copies of tables no longer available in Neon are retained.
- All import operations and results are documented.

### Import and Inventory Process

1. Establish a connection to the Neon database.
2. Retrieve the available database tables and column information.
3. Scan the local original data directory.
4. Compare database tables with existing local copies.
5. Display the current database and local data status.
6. Import missing local copies without overwriting existing files.
7. Create or update the required working copies.
8. Display an import summary, including skipped tables, newly imported tables, and errors.

### Section Overview

- **4.0 Database Configuration and Inventory**
- **4.1 Local Data Inventory and Comparison**
- **4.2 Import Data and Create Working Copies**

In [14]:
# 4.0 Database Configuration and Inventory

import os
import psycopg
from dotenv import load_dotenv

# Initialize database status
database_connected = False
database_info_available = False
database_tables = []
database_columns = []
connection_parameters = {}

# Load environment variables
load_dotenv(override=True)

# Define required environment variables
required_variables = [
    "DB_HOST",
    "DB_PORT",
    "DB_NAME",
    "DB_USER",
    "DB_PASSWORD"
]

# Check database configuration
missing_variables = [
    variable
    for variable in required_variables
    if not os.getenv(variable)
]

print("TRAVELTIDE – DATABASE CONFIGURATION")
print("=" * 60)

for variable in required_variables:
    status = "OK" if variable not in missing_variables else "MISSING"
    print(f"{variable}: {status}")

if missing_variables:
    print("\nDatabase connection: SKIPPED")
    print("Missing variables:", ", ".join(missing_variables))

else:
    # Build database connection parameters
    connection_parameters = {
        "host": os.getenv("DB_HOST"),
        "port": os.getenv("DB_PORT"),
        "dbname": os.getenv("DB_NAME"),
        "user": os.getenv("DB_USER"),
        "password": os.getenv("DB_PASSWORD"),
        "sslmode": "require",
        "connect_timeout": 10
    }

    try:
        with psycopg.connect(**connection_parameters) as connection:
            with connection.cursor() as cursor:

                # Retrieve PostgreSQL version
                cursor.execute("SELECT version();")
                database_version = cursor.fetchone()[0]

                # Retrieve database table and column information
                cursor.execute("""
                    SELECT
                        table_schema,
                        table_name,
                        column_name,
                        ordinal_position,
                        data_type,
                        udt_name
                    FROM information_schema.columns
                    WHERE table_schema NOT IN (
                        'pg_catalog',
                        'information_schema'
                    )
                    AND table_schema NOT LIKE 'pg_toast%'
                    ORDER BY
                        table_schema,
                        table_name,
                        ordinal_position;
                """)

                database_columns = cursor.fetchall()

        # Create a unique list of database tables
        database_tables = sorted({
            (row[0], row[1])
            for row in database_columns
        })

        database_connected = True
        database_info_available = True

        print("\nDatabase connection: SUCCESSFUL")
        print("PostgreSQL version:", database_version)
        print(f"Database tables found: {len(database_tables)}")

        # Display database table inventory
        for schema, table in database_tables:
            print(f"\n{schema}.{table}")

            table_columns = [
                row for row in database_columns
                if row[0] == schema and row[1] == table
            ]

            for row in table_columns:
                print(
                    f"  {row[3]:>2}. "
                    f"{row[2]:<30} "
                    f"{row[4]} ({row[5]})"
                )

    except psycopg.Error as error:
        database_connected = False
        database_info_available = False
        database_tables = []
        database_columns = []

        print("\nDatabase connection or inventory: FAILED")
        print("Error:", error)

print("=" * 60)


TRAVELTIDE – DATABASE CONFIGURATION
DB_HOST: OK
DB_PORT: OK
DB_NAME: OK
DB_USER: OK
DB_PASSWORD: OK

Database connection: SUCCESSFUL
PostgreSQL version: PostgreSQL 15.19 (f55ec65) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 10.2.1-6) 10.2.1 20210110, 64-bit
Database tables found: 4

public.flights
   1. trip_id                        text (text)
   2. origin_airport                 text (text)
   3. destination                    text (text)
   4. destination_airport            text (text)
   5. seats                          integer (int4)
   6. return_flight_booked           boolean (bool)
   7. departure_time                 timestamp without time zone (timestamp)
   8. return_time                    timestamp without time zone (timestamp)
   9. checked_bags                   integer (int4)
  10. trip_airline                   text (text)
  11. destination_airport_lat        numeric (numeric)
  12. destination_airport_lon        numeric (numeric)
  13. base_fare_usd       

### 4.1 Local Data Inventory and Comparison

This section examines the locally available original CSV files and compares them with the tables stored in the Neon PostgreSQL database.

The local inventory provides an overview of file names, formats, file sizes, and modification dates. The comparison identifies database tables that are missing locally and local files that no longer have a corresponding database table.

This verification ensures that all required original datasets are available before the import and working-copy process begins.

In [15]:

# 4.1 Local Data Inventory and Comparison

import pandas as pd
from pathlib import Path

# Define the original data directory
ORIGINAL_DIR = Path("..") / "data" / "input" / "original"
ORIGINAL_DIR = ORIGINAL_DIR.resolve()

print("TRAVELTIDE – LOCAL DATA INVENTORY AND COMPARISON")
print("=" * 60)

# Scan existing local original CSV files
original_files = sorted(ORIGINAL_DIR.glob("*.csv"))

local_inventory_data = [
    {
        "File Name": file.name,
        "File Format": file.suffix.lower(),
        "File Size (KB)": round(file.stat().st_size / 1024, 2),
        "Last Modified": pd.to_datetime(
            file.stat().st_mtime, unit="s"
        )
    }
    for file in original_files
]

# Create and display local inventory
local_inventory = pd.DataFrame(
    local_inventory_data,
    columns=[
        "File Name",
        "File Format",
        "File Size (KB)",
        "Last Modified"
    ]
)

print(f"\nLocal original copies found: {len(local_inventory)}")
display(local_inventory)

# Compare database tables with local original copies
if not database_info_available:

    print("\nDatabase comparison: SKIPPED")
    print("Reason: Database table information is unavailable.")

    existing_locally = set()
    missing_locally = set()
    missing_in_database = set()

else:
    local_table_names = {
        file.stem.replace("__", ".", 1)
        for file in original_files
    }

    database_table_names = {
        f"{schema}.{table}"
        for schema, table in database_tables
    }

    existing_locally = database_table_names & local_table_names
    missing_locally = database_table_names - local_table_names
    missing_in_database = local_table_names - database_table_names

    print("\nDATABASE AND LOCAL DATA COMPARISON")
    print("-" * 60)

    print(f"Tables available in Neon: {len(database_table_names)}")
    print(f"Local original copies: {len(local_table_names)}")
    print(f"Existing locally: {len(existing_locally)}")
    print(f"Missing locally: {len(missing_locally)}")
    print(f"Missing in Neon: {len(missing_in_database)}")

    if missing_locally:
        print("\nTables requiring download:")
        for table in sorted(missing_locally):
            print(f"- {table}")

    if missing_in_database:
        print("\nLocal copies without a database table:")
        for table in sorted(missing_in_database):
            print(f"- {table}")

    print("\nDatabase connection:", "CONNECTED" if database_connected else "NOT CONNECTED")
    print("Database inventory:", "AVAILABLE" if database_info_available else "NOT AVAILABLE")

print("=" * 60)


TRAVELTIDE – LOCAL DATA INVENTORY AND COMPARISON

Local original copies found: 4


,File Name,File Format,File Size (KB),Last Modified
0,public__flights.csv,.csv,258929.20,2026-11-01 03:52:57.000549316
1,public__hotels.csv,.csv,211123.80,2026-11-01 03:53:21.363658667
2,public__sessions.csv,.csv,634810.70,2026-11-01 03:54:50.630051374
3,public__users.csv,.csv,67760.92,2026-11-01 03:54:58.960024118



DATABASE AND LOCAL DATA COMPARISON
------------------------------------------------------------
Tables available in Neon: 4
Local original copies: 4
Existing locally: 4
Missing locally: 0
Missing in Neon: 0

Database connection: CONNECTED
Database inventory: AVAILABLE


### 4.2 Import Data and Create Working Copies

This section imports database tables from Neon PostgreSQL as local CSV files, if the corresponding original files do not already exist.

Existing original files are preserved and never overwritten. Database access is restricted to read-only operations to protect the source data.

Once all required original files are available and no import errors have occurred, working copies are created or updated when necessary. These copies are stored separately in the `data/working/` directory and serve as the basis for subsequent data cleaning, transformations, and analysis.

If all original files and working copies are already available, no changes are made.

In [16]:
# 4.2 Import Data and Create Working Copies

from psycopg import sql
import shutil

print("TRAVELTIDE – DATA IMPORT AND WORKING COPIES")
print("=" * 60)

# Initialize import results
imported_tables = []
skipped_tables = []
failed_tables = []
working_copies_created = []

# Define the working data directory
WORKING_DIR = ORIGINAL_DIR.parent.parent / "working"

# Define required source files
required_tables = [
    "public__flights.csv",
    "public__hotels.csv",
    "public__sessions.csv",
    "public__users.csv"
]

# Check database availability
if not database_connected or not database_info_available:

    print("Import: SKIPPED")
    print("Reason: Database connection or inventory unavailable.")

else:
    try:
        with psycopg.connect(**connection_parameters) as connection:

            # Enforce read-only database access
            with connection.cursor() as cursor:
                cursor.execute("SET TRANSACTION READ ONLY")

                for schema, table in database_tables:

                    file_name = f"{schema}__{table}.csv"
                    target_file = ORIGINAL_DIR / file_name

                    # Preserve existing original copies
                    if target_file.exists():
                        skipped_tables.append(file_name)
                        print(f"SKIPPED: {file_name}")
                        continue

                    try:
                        # Export the database table without overwriting files
                        with target_file.open("xb") as output_file:

                            copy_query = sql.SQL(
                                "COPY {}.{} TO STDOUT "
                                "WITH (FORMAT CSV, HEADER TRUE, ENCODING 'UTF8')"
                            ).format(
                                sql.Identifier(schema),
                                sql.Identifier(table)
                            )

                            with cursor.copy(copy_query) as copy:
                                for data in copy:
                                    output_file.write(data)

                        imported_tables.append(file_name)
                        print(f"IMPORTED: {file_name}")

                    except FileExistsError:
                        skipped_tables.append(file_name)
                        print(f"SKIPPED: {file_name}")

                    except Exception as error:
                        if target_file.exists():
                            target_file.unlink()

                        failed_tables.append(file_name)
                        print(f"FAILED: {file_name} – {error}")

    except psycopg.Error as error:
        failed_tables.append("DATABASE CONNECTION OR IMPORT")
        print("Database import failed:", error)

    # Check whether all required original files exist
    missing_originals = [
        file_name
        for file_name in required_tables
        if not (ORIGINAL_DIR / file_name).is_file()
    ]

    # Create working copies only if all originals are available
    if not failed_tables and not missing_originals:

        WORKING_DIR.mkdir(parents=True, exist_ok=True)

        existing_working_copies = [
            file_name
            for file_name in required_tables
            if not (WORKING_DIR / file_name).is_file()
        ]

        # Recreate all working copies when necessary
        if imported_tables or existing_working_copies:

            for file_name in required_tables:
                source_file = ORIGINAL_DIR / file_name
                target_file = WORKING_DIR / file_name

                shutil.copy2(source_file, target_file)
                working_copies_created.append(file_name)

            print("\nWORKING COPIES")
            print("-" * 60)

            for file_name in working_copies_created:
                print(f"CREATED: {file_name}")

        else:
            print("\nWorking copies: No changes required.")

    else:
        print("\nWORKING COPIES: NOT CREATED")

        if missing_originals:
            print("Missing original files:", ", ".join(missing_originals))

        if failed_tables:
            print("Import errors:", ", ".join(failed_tables))

# Display import summary
print("\nIMPORT SUMMARY")
print("-" * 60)
print(f"New local copies: {len(imported_tables)}")
print(f"Existing copies skipped: {len(skipped_tables)}")
print(f"Failed imports: {len(failed_tables)}")
print(f"Working copies created: {len(working_copies_created)}")

if failed_tables:
    print("Import status: COMPLETED WITH ERRORS")
elif not database_connected or not database_info_available:
    print("Import status: NOT PERFORMED")
elif not imported_tables and not skipped_tables:
    print("Import status: COMPLETED – NO TABLES FOUND")
else:
    print("Import status: COMPLETED")

print("=" * 60)


TRAVELTIDE – DATA IMPORT AND WORKING COPIES
SKIPPED: public__flights.csv
SKIPPED: public__hotels.csv
SKIPPED: public__sessions.csv
SKIPPED: public__users.csv

Working copies: No changes required.

IMPORT SUMMARY
------------------------------------------------------------
New local copies: 0
Existing copies skipped: 4
Failed imports: 0
Working copies created: 0
Import status: COMPLETED
